# Notebook 01 — Per-Product Distributional Analysis

**Assignment**: Explorer 01 — Distributional family  
**Lenses**: mid stats, distinct prices, hardcoded-FV check, range/CV, mode prevalence  
**Data**: `prices_round_5_day_{2,3,4}.csv` (read-only, semicolon-separated)  
**Scope**: all 50 products × days 2/3/4

## Cell 1 — Imports and data loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path

DATA_DIR = Path('../../data/round_5/prices')

dfs = []
for day in [2, 3, 4]:
    df = pd.read_csv(DATA_DIR / f'prices_round_5_day_{day}.csv', sep=';')
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(all_df):,}')
print(f'Products  : {all_df["product"].nunique()}')
print(f'Days      : {sorted(all_df["day"].unique())}')
print(f'Columns   : {list(all_df.columns)}')

Total rows: 1,500,000
Products  : 50
Days      : [np.int64(2), np.int64(3), np.int64(4)]
Columns   : ['day', 'timestamp', 'product', 'bid_price_1', 'bid_volume_1', 'bid_price_2', 'bid_volume_2', 'bid_price_3', 'bid_volume_3', 'ask_price_1', 'ask_volume_1', 'ask_price_2', 'ask_volume_2', 'ask_price_3', 'ask_volume_3', 'mid_price', 'profit_and_loss']


## Cell 2 — Mid-price summary stats (mean, std, min, max, median) per product × day

In [2]:
mid_stats = (
    all_df.groupby(['product', 'day'])['mid_price']
    .agg(['mean', 'std', 'min', 'max', 'median', 'count'])
    .rename(columns={'count': 'n_ticks'})
    .reset_index()
)

# Pivot for compact display — show mean across days
mean_pivot = mid_stats.pivot(index='product', columns='day', values='mean').round(2)
std_pivot  = mid_stats.pivot(index='product', columns='day', values='std').round(4)

print('=== Mean mid-price by product × day ===')
print(mean_pivot.to_string())
print()
print('=== Std dev of mid-price by product × day ===')
print(std_pivot.to_string())

=== Mean mid-price by product × day ===
day                                   2         3         4
product                                                    
GALAXY_SOUNDS_BLACK_HOLES      10680.49  11107.88  12612.25
GALAXY_SOUNDS_DARK_MATTER      10111.91  10421.95  10146.12
GALAXY_SOUNDS_PLANETARY_RINGS  10013.25  11615.27  10671.49
GALAXY_SOUNDS_SOLAR_FLAMES     11095.64  11260.00  10922.07
GALAXY_SOUNDS_SOLAR_WINDS      10066.58  10309.85  10936.20
MICROCHIP_CIRCLE                9190.40   8897.68   9556.57
MICROCHIP_OVAL                  9766.38   8543.77   6228.64
MICROCHIP_RECTANGLE             9596.87   8293.35   8307.09
MICROCHIP_SQUARE               11268.46  14584.24  14931.54
MICROCHIP_TRIANGLE             10216.49  10229.08   8613.60
OXYGEN_SHAKE_CHOCOLATE          9355.81   9243.59  10071.24
OXYGEN_SHAKE_EVENING_BREATH     9184.56   9204.29   9426.83
OXYGEN_SHAKE_GARLIC            11058.25  11807.84  12910.83
OXYGEN_SHAKE_MINT               9893.52  10265.45   9356.21


## Cell 3 — Distinct prices and mode prevalence

In [3]:
def distinct_and_mode(grp):
    vc = grp['mid_price'].value_counts()
    n = len(grp)
    mode_val  = vc.index[0]
    mode_cnt  = vc.iloc[0]
    mode_pct  = mode_cnt / n
    n_distinct = len(vc)
    return pd.Series({
        'n_ticks'   : n,
        'n_distinct': n_distinct,
        'mode_val'  : mode_val,
        'mode_count': mode_cnt,
        'mode_pct'  : round(mode_pct, 6),
    })

dm = all_df.groupby(['product', 'day']).apply(distinct_and_mode, include_groups=False).reset_index()

# Products where the single mode_val accounts for >=50% of all ticks in any day
high_mode = dm[dm['mode_pct'] >= 0.50].sort_values('mode_pct', ascending=False)
print('Products where mode_val ≥50% of ticks (any day):')
print(high_mode[['product','day','mode_val','mode_count','n_ticks','mode_pct']].to_string(index=False))
print()

# Full table sorted by mode_pct descending, first-day row per product
print('=== All products: mode prevalence (sorted) ===')
avg_mode_pct = dm.groupby('product')['mode_pct'].mean().sort_values(ascending=False)
print(avg_mode_pct.round(4).to_string())

Products where mode_val ≥50% of ticks (any day):
Empty DataFrame
Columns: [product, day, mode_val, mode_count, n_ticks, mode_pct]
Index: []

=== All products: mode prevalence (sorted) ===
product
ROBOT_DISHES                     0.0563
OXYGEN_SHAKE_EVENING_BREATH      0.0419
ROBOT_IRONING                    0.0339
OXYGEN_SHAKE_CHOCOLATE           0.0112
SNACKPACK_PISTACHIO              0.0046
SNACKPACK_VANILLA                0.0039
SNACKPACK_CHOCOLATE              0.0038
SNACKPACK_STRAWBERRY             0.0036
SNACKPACK_RASPBERRY              0.0036
TRANSLATOR_GRAPHITE_MIST         0.0034
TRANSLATOR_ASTRO_BLACK           0.0032
PANEL_2X2                        0.0032
TRANSLATOR_VOID_BLUE             0.0029
UV_VISOR_AMBER                   0.0028
ROBOT_VACUUMING                  0.0028
GALAXY_SOUNDS_DARK_MATTER        0.0027
UV_VISOR_RED                     0.0026
MICROCHIP_CIRCLE                 0.0026
ROBOT_LAUNDRY                    0.0026
UV_VISOR_YELLOW                  0.0026
PANE

## Cell 4 — Hardcoded Fair Value detection

A product has a "hardcoded FV" if one price appears ≥90% of ticks across all 3 days combined.

In [4]:
def fv_check(grp):
    vc = grp['mid_price'].value_counts()
    n = len(grp)
    mode_val = vc.index[0]
    mode_pct = vc.iloc[0] / n
    # Check top-2 FVs
    top2 = vc.head(2)
    combined_pct = top2.sum() / n
    return pd.Series({
        'n_ticks'       : n,
        'fv_candidate'  : mode_val,
        'fv_pct'        : round(mode_pct, 6),
        'top2_combined' : round(combined_pct, 6),
        'hardcoded_FV'  : mode_pct >= 0.90,
    })

fv = all_df.groupby('product').apply(fv_check, include_groups=False).reset_index()
fv_sorted = fv.sort_values('fv_pct', ascending=False)

print('=== Hardcoded-FV check (ALL 3 days combined, sorted by fv_pct) ===')
print(fv_sorted.to_string(index=False))
print()
print('Products flagged as hardcoded_FV (fv_pct >= 0.90):')
print(fv_sorted[fv_sorted['hardcoded_FV']][['product','fv_candidate','fv_pct','n_ticks']].to_string(index=False))

=== Hardcoded-FV check (ALL 3 days combined, sorted by fv_pct) ===
                      product  n_ticks  fv_candidate   fv_pct  top2_combined  hardcoded_FV
                 ROBOT_DISHES    30000       10600.0 0.053800       0.092400         False
  OXYGEN_SHAKE_EVENING_BREATH    30000       10100.0 0.022733       0.036533         False
                ROBOT_IRONING    30000       10000.0 0.017700       0.027867         False
       OXYGEN_SHAKE_CHOCOLATE    30000        9250.0 0.006467       0.012900         False
            SNACKPACK_VANILLA    30000        9996.5 0.002833       0.005667         False
          SNACKPACK_CHOCOLATE    30000       10005.5 0.002733       0.005367         False
          SNACKPACK_RASPBERRY    30000        9922.5 0.002700       0.005267         False
          SNACKPACK_PISTACHIO    30000        9378.0 0.002633       0.005233         False
     TRANSLATOR_GRAPHITE_MIST    30000        9962.5 0.002067       0.003967         False
    GALAXY_SOUNDS_DARK_

## Cell 5 — Hardcoded FV check: per-day stability

Verify that the FV candidate is consistent across days (not an artifact of one day).

In [5]:
def fv_per_day(grp):
    vc = grp['mid_price'].value_counts()
    n = len(grp)
    return pd.Series({
        'n_ticks'  : n,
        'fv_val'   : vc.index[0],
        'fv_pct'   : round(vc.iloc[0] / n, 6),
        'n_distinct': len(vc),
    })

fv_day = all_df.groupby(['product','day']).apply(fv_per_day, include_groups=False).reset_index()

# Focus on products with fv_pct >= 0.80 on any day
high_fv_day = fv_day[fv_day['fv_pct'] >= 0.80].sort_values(['product','day'])
print('Products with per-day fv_pct >= 0.80:')
print(high_fv_day.to_string(index=False))
print()

# Pivot to see fv_val across days — check if the candidate is same on all 3 days
fv_val_pivot = fv_day.pivot(index='product', columns='day', values='fv_val')
fv_pct_pivot = fv_day.pivot(index='product', columns='day', values='fv_pct')
# Same FV on all 3 days?
same_fv = (fv_val_pivot.iloc[:,0] == fv_val_pivot.iloc[:,1]) & (fv_val_pivot.iloc[:,1] == fv_val_pivot.iloc[:,2])
print('FV candidate is same across all 3 days:')
print(pd.DataFrame({'fv_same': same_fv, 'fv_day2': fv_val_pivot[2], 'fv_day3': fv_val_pivot[3], 'fv_day4': fv_val_pivot[4],
                    'pct_day2': fv_pct_pivot[2].round(4), 'pct_day3': fv_pct_pivot[3].round(4), 'pct_day4': fv_pct_pivot[4].round(4)
                    }).sort_values('pct_day2', ascending=False).to_string())

Products with per-day fv_pct >= 0.80:
Empty DataFrame
Columns: [product, day, n_ticks, fv_val, fv_pct, n_distinct]
Index: []

FV candidate is same across all 3 days:
                               fv_same  fv_day2  fv_day3  fv_day4  pct_day2  pct_day3  pct_day4
product                                                                                        
OXYGEN_SHAKE_EVENING_BREATH      False  10100.0   9220.0   9380.0    0.0682    0.0303    0.0272
ROBOT_IRONING                    False  10000.0   9330.0   7650.0    0.0531    0.0196    0.0289
OXYGEN_SHAKE_CHOCOLATE           False   9250.0   9053.0  10900.0    0.0181    0.0029    0.0127
SNACKPACK_PISTACHIO              False   9549.0   9334.0   9427.0    0.0046    0.0037    0.0056
SNACKPACK_CHOCOLATE              False  10004.5   9952.5   9629.0    0.0045    0.0044    0.0026
TRANSLATOR_ASTRO_BLACK           False  10014.5   8790.0   9274.0    0.0045    0.0028    0.0024
SNACKPACK_VANILLA                False   9996.5   9985.5  10207.5 

## Cell 6 — Range and Coefficient of Variation (CV) per product × day

In [6]:
def range_cv(grp):
    m = grp['mid_price']
    price_range = m.max() - m.min()
    mean_val = m.mean()
    std_val  = m.std()
    cv = std_val / mean_val if mean_val != 0 else np.nan
    return pd.Series({
        'mean'       : round(mean_val, 4),
        'std'        : round(std_val, 4),
        'range'      : round(price_range, 4),
        'cv'         : round(cv, 8),
        'range_pct'  : round(price_range / mean_val, 6) if mean_val != 0 else np.nan,
    })

rv = all_df.groupby(['product','day']).apply(range_cv, include_groups=False).reset_index()

# Average CV across days per product
cv_avg = rv.groupby('product')['cv'].mean().sort_values()
print('=== Average CV (low = tighter / more hardcoded) ===')
print(cv_avg.round(8).to_string())
print()

# Average range_pct across days per product
rng_avg = rv.groupby('product')['range_pct'].mean().sort_values()
print('=== Average range_pct (range / mean) ===')
print(rng_avg.round(6).to_string())

=== Average CV (low = tighter / more hardcoded) ===
product
SNACKPACK_PISTACHIO              0.014482
SNACKPACK_CHOCOLATE              0.015784
SNACKPACK_VANILLA                0.016011
SNACKPACK_RASPBERRY              0.016572
SNACKPACK_STRAWBERRY             0.018081
ROBOT_LAUNDRY                    0.024670
UV_VISOR_RED                     0.025426
ROBOT_VACUUMING                  0.027799
ROBOT_DISHES                     0.028054
GALAXY_SOUNDS_DARK_MATTER        0.029144
TRANSLATOR_ASTRO_BLACK           0.029576
PANEL_2X2                        0.029947
UV_VISOR_MAGENTA                 0.030076
TRANSLATOR_VOID_BLUE             0.030200
TRANSLATOR_ECLIPSE_CHARCOAL      0.031780
OXYGEN_SHAKE_MINT                0.032368
SLEEP_POD_NYLON                  0.032708
UV_VISOR_ORANGE                  0.033446
SLEEP_POD_SUEDE                  0.034265
SLEEP_POD_LAMB_WOOL              0.034671
UV_VISOR_AMBER                   0.034773
MICROCHIP_TRIANGLE               0.035596
PANEL_2X4       

## Cell 7 — Mid-price distribution plots: violin + per-product histogram grids

One violin plot per category, showing all 3 days side-by-side.

In [7]:
import warnings
warnings.filterwarnings('ignore')

CATEGORIES = {
    'GALAXY_SOUNDS' : ['GALAXY_SOUNDS_BLACK_HOLES','GALAXY_SOUNDS_DARK_MATTER','GALAXY_SOUNDS_PLANETARY_RINGS','GALAXY_SOUNDS_SOLAR_FLAMES','GALAXY_SOUNDS_SOLAR_WINDS'],
    'SLEEP_POD'     : ['SLEEP_POD_COTTON','SLEEP_POD_LAMB_WOOL','SLEEP_POD_NYLON','SLEEP_POD_POLYESTER','SLEEP_POD_SUEDE'],
    'MICROCHIP'     : ['MICROCHIP_CIRCLE','MICROCHIP_OVAL','MICROCHIP_RECTANGLE','MICROCHIP_SQUARE','MICROCHIP_TRIANGLE'],
    'PEBBLES'       : ['PEBBLES_XS','PEBBLES_S','PEBBLES_M','PEBBLES_L','PEBBLES_XL'],
    'ROBOT'         : ['ROBOT_DISHES','ROBOT_IRONING','ROBOT_LAUNDRY','ROBOT_MOPPING','ROBOT_VACUUMING'],
    'UV_VISOR'      : ['UV_VISOR_AMBER','UV_VISOR_MAGENTA','UV_VISOR_ORANGE','UV_VISOR_RED','UV_VISOR_YELLOW'],
    'TRANSLATOR'    : ['TRANSLATOR_ASTRO_BLACK','TRANSLATOR_ECLIPSE_CHARCOAL','TRANSLATOR_GRAPHITE_MIST','TRANSLATOR_SPACE_GRAY','TRANSLATOR_VOID_BLUE'],
    'PANEL'         : ['PANEL_1X2','PANEL_1X4','PANEL_2X2','PANEL_2X4','PANEL_4X4'],
    'OXYGEN_SHAKE'  : ['OXYGEN_SHAKE_CHOCOLATE','OXYGEN_SHAKE_EVENING_BREATH','OXYGEN_SHAKE_GARLIC','OXYGEN_SHAKE_MINT','OXYGEN_SHAKE_MORNING_BREATH'],
    'SNACKPACK'     : ['SNACKPACK_CHOCOLATE','SNACKPACK_PISTACHIO','SNACKPACK_RASPBERRY','SNACKPACK_STRAWBERRY','SNACKPACK_VANILLA'],
}

DAYS = [2, 3, 4]
COLORS = {2: 'steelblue', 3: 'darkorange', 4: 'forestgreen'}

plots_dir = Path('plots')
plots_dir.mkdir(exist_ok=True)

for cat_name, products in CATEGORIES.items():
    fig, axes = plt.subplots(1, len(products), figsize=(18, 4), sharey=False)
    fig.suptitle(f'{cat_name} — mid-price distribution (days 2/3/4)', fontsize=13)
    for ax, prod in zip(axes, products):
        data = [all_df[(all_df['product']==prod) & (all_df['day']==d)]['mid_price'].values for d in DAYS]
        parts = ax.violinplot(data, positions=[0,1,2], showmedians=True, widths=0.7)
        for pc, d in zip(parts['bodies'], DAYS):
            pc.set_facecolor(COLORS[d])
            pc.set_alpha(0.6)
        short = prod.replace(cat_name+'_', '')
        ax.set_title(short, fontsize=9)
        ax.set_xticks([0,1,2])
        ax.set_xticklabels(['D2','D3','D4'], fontsize=8)
        ax.tick_params(axis='y', labelsize=7)
    plt.tight_layout()
    out_path = plots_dir / f'01_violin_{cat_name.lower()}.png'
    plt.savefig(out_path, dpi=100)
    plt.show()
    plt.close()
    print(f'Saved {out_path}')

Saved plots/01_violin_galaxy_sounds.png


Saved plots/01_violin_sleep_pod.png


Saved plots/01_violin_microchip.png


Saved plots/01_violin_pebbles.png


Saved plots/01_violin_robot.png


Saved plots/01_violin_uv_visor.png


Saved plots/01_violin_translator.png


Saved plots/01_violin_panel.png


Saved plots/01_violin_oxygen_shake.png


Saved plots/01_violin_snackpack.png


## Cell 8 — Hardcoded FV products: histogram of mid_price values

For products with very high mode prevalence, plot the full value-count distribution.

In [8]:
# Products with avg fv_pct >= 0.90 across days
hfv_products = fv[fv['fv_pct'] >= 0.90]['product'].tolist()
print(f'Products with all-days fv_pct >= 0.90: {hfv_products}')

# Also flag products with avg mode_pct (per-day average) >= 0.50
avg_mode = dm.groupby('product')['mode_pct'].mean()
moderate_pin = avg_mode[avg_mode >= 0.50].index.tolist()
print(f'Products with avg mode_pct >= 0.50: {sorted(moderate_pin)}')

focus_products = sorted(set(hfv_products + moderate_pin))
print(f'Focus products (union): {focus_products}')

if focus_products:
    n_cols = 3
    n_rows = int(np.ceil(len(focus_products) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = np.array(axes).flatten()
    for ax, prod in zip(axes, focus_products):
        for d, color in COLORS.items():
            sub = all_df[(all_df['product']==prod) & (all_df['day']==d)]['mid_price']
            vc = sub.value_counts().sort_index()
            ax.bar(vc.index, vc.values, width=0.5, alpha=0.5, label=f'D{d}', color=color)
        ax.set_title(prod, fontsize=8)
        ax.legend(fontsize=7)
        ax.tick_params(labelsize=7)
    for ax in axes[len(focus_products):]:
        ax.set_visible(False)
    plt.suptitle('Mid-price value counts — high-mode products', fontsize=12)
    plt.tight_layout()
    plt.savefig(plots_dir / '01_high_mode_histograms.png', dpi=100)
    plt.show()
    plt.close()
else:
    print('No products with avg mode_pct >= 0.50 — plotting all products in grid instead')
    products_all = sorted(all_df['product'].unique())
    n_cols = 5
    n_rows = int(np.ceil(len(products_all) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
    axes = axes.flatten()
    for ax, prod in zip(axes, products_all):
        for d, color in COLORS.items():
            sub = all_df[(all_df['product']==prod) & (all_df['day']==d)]['mid_price']
            sub.hist(bins=50, ax=ax, alpha=0.4, color=color, label=f'D{d}')
        ax.set_title(prod.replace('_',' '), fontsize=6)
        ax.tick_params(labelsize=5)
    for ax in axes[len(products_all):]:
        ax.set_visible(False)
    plt.suptitle('Mid-price histograms — all 50 products', fontsize=12)
    plt.tight_layout()
    plt.savefig(plots_dir / '01_all_histograms.png', dpi=100)
    plt.show()
    plt.close()

Products with all-days fv_pct >= 0.90: []
Products with avg mode_pct >= 0.50: []
Focus products (union): []
No products with avg mode_pct >= 0.50 — plotting all products in grid instead


## Cell 9 — Category-level CV and range comparison

Summary heatmap of CV across categories, highlighting which categories are tightly pinned.

In [9]:
# Add category column
product_to_cat = {}
for cat, prods in CATEGORIES.items():
    for p in prods:
        product_to_cat[p] = cat

rv['category'] = rv['product'].map(product_to_cat)

cat_cv = rv.groupby(['category', 'day'])['cv'].mean().unstack('day').round(8)
cat_range = rv.groupby(['category', 'day'])['range_pct'].mean().unstack('day').round(6)

print('=== Mean CV by category × day ===')
print(cat_cv.to_string())
print()
print('=== Mean range_pct by category × day ===')
print(cat_range.to_string())

# Plot heatmap of mean CV per category (averaged over days and products)
cv_overall = rv.groupby('category')['cv'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cv_overall.index, cv_overall.values, color='steelblue')
ax.set_xlabel('Mean CV (std/mean of mid_price)')
ax.set_title('Category-level mean CV — lower = more tightly pinned')
for bar, val in zip(bars, cv_overall.values):
    ax.text(val + cv_overall.max()*0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.6f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(plots_dir / '01_category_cv.png', dpi=100)
plt.show()
plt.close()
print('Saved 01_category_cv.png')

=== Mean CV by category × day ===
day                   2         3         4
category                                   
GALAXY_SOUNDS  0.041392  0.034030  0.033879
MICROCHIP      0.036574  0.054788  0.058157
OXYGEN_SHAKE   0.040336  0.034242  0.043051
PANEL          0.034914  0.045391  0.037928
PEBBLES        0.064749  0.058553  0.068284
ROBOT          0.031188  0.049080  0.024331
SLEEP_POD      0.042807  0.034166  0.032254
SNACKPACK      0.016683  0.015458  0.016417
TRANSLATOR     0.031475  0.034653  0.035759
UV_VISOR       0.039595  0.026772  0.038893

=== Mean range_pct by category × day ===
day                   2         3         4
category                                   
GALAXY_SOUNDS  0.174392  0.147811  0.151843
MICROCHIP      0.168306  0.229097  0.242854
OXYGEN_SHAKE   0.158050  0.148148  0.166516
PANEL          0.156614  0.177063  0.167445
PEBBLES        0.257028  0.250318  0.282261
ROBOT          0.134280  0.183702  0.111210
SLEEP_POD      0.177438  0.144053  0.136175


## Cell 10 — Per-product distinct-price count and discreteness analysis

In [10]:
# Number of distinct mid-price values per product (all days combined)
distinct_all = all_df.groupby('product')['mid_price'].nunique().sort_values()
print('=== Distinct mid-price values (all days) ===')
print(distinct_all.to_string())
print(f'\nMedian distinct prices: {distinct_all.median()}')
print(f'Min: {distinct_all.min()} ({distinct_all.idxmin()})  Max: {distinct_all.max()} ({distinct_all.idxmax()})')

# Per-day distinct
distinct_day = all_df.groupby(['product','day'])['mid_price'].nunique().unstack('day')
print()
print('=== Distinct mid-price values per day ===')
print(distinct_day.sort_values(2).to_string())

# Plot distinct-price histogram
fig, ax = plt.subplots(figsize=(12, 5))
colors_bar = [COLORS[2]] * len(distinct_all)
ax.bar(range(len(distinct_all)), distinct_all.values, color='steelblue', alpha=0.7)
ax.set_xticks(range(len(distinct_all)))
ax.set_xticklabels(distinct_all.index, rotation=90, fontsize=6)
ax.set_ylabel('# distinct mid-price values')
ax.set_title('Distinct mid-price values per product (all days combined)')
plt.tight_layout()
plt.savefig(plots_dir / '01_distinct_prices.png', dpi=100)
plt.show()
plt.close()

=== Distinct mid-price values (all days) ===
product
OXYGEN_SHAKE_EVENING_BREATH        453
ROBOT_IRONING                      631
SNACKPACK_RASPBERRY               1634
SNACKPACK_VANILLA                 1673
SNACKPACK_CHOCOLATE               1900
SNACKPACK_PISTACHIO               1928
GALAXY_SOUNDS_DARK_MATTER         2907
ROBOT_DISHES                      3048
SNACKPACK_STRAWBERRY              3089
TRANSLATOR_ECLIPSE_CHARCOAL       3146
SLEEP_POD_LAMB_WOOL               3428
OXYGEN_SHAKE_CHOCOLATE            3512
TRANSLATOR_ASTRO_BLACK            3560
ROBOT_VACUUMING                   3611
TRANSLATOR_SPACE_GRAY             3722
GALAXY_SOUNDS_SOLAR_FLAMES        3730
PANEL_4X4                         3841
ROBOT_LAUNDRY                     3912
TRANSLATOR_GRAPHITE_MIST          3937
OXYGEN_SHAKE_MINT                 3973
SLEEP_POD_NYLON                   4004
MICROCHIP_CIRCLE                  4018
UV_VISOR_ORANGE                   4130
TRANSLATOR_VOID_BLUE              4135
GALAXY_SOUN


=== Distinct mid-price values per day ===
day                               2     3     4
product                                        
OXYGEN_SHAKE_CHOCOLATE          277  1636  2835
ROBOT_IRONING                   302   425   238
OXYGEN_SHAKE_EVENING_BREATH     344   276   266
SNACKPACK_PISTACHIO            1123  1098  1058
SNACKPACK_CHOCOLATE            1178  1344  1359
SNACKPACK_VANILLA              1408  1273  1315
SNACKPACK_RASPBERRY            1415  1266  1348
UV_VISOR_ORANGE                1566  2172  3012
TRANSLATOR_ASTRO_BLACK         1582  2367  1764
OXYGEN_SHAKE_MINT              1634  1728  2861
PANEL_4X4                      1637  2235  3107
PANEL_2X2                      1662  2437  1708
GALAXY_SOUNDS_DARK_MATTER      1785  2228  2193
TRANSLATOR_GRAPHITE_MIST       1815  2324  2448
SNACKPACK_STRAWBERRY           1825  1372  1245
ROBOT_VACUUMING                1842  2421  1236
ROBOT_LAUNDRY                  1861  2124  1573
MICROCHIP_CIRCLE               1865  1997  31

## Cell 11 — Mode prevalence heatmap across all 50 products × 3 days

In [11]:
mode_pct_pivot = dm.pivot(index='product', columns='day', values='mode_pct')
mode_pct_avg = mode_pct_pivot.mean(axis=1).sort_values(ascending=False)

# Reorder by avg
mode_pct_pivot_sorted = mode_pct_pivot.loc[mode_pct_avg.index]

fig, ax = plt.subplots(figsize=(8, 14))
im = ax.imshow(mode_pct_pivot_sorted.values, aspect='auto', cmap='RdYlGn',
               vmin=0, vmax=1, interpolation='nearest')
ax.set_xticks([0,1,2])
ax.set_xticklabels(['Day 2','Day 3','Day 4'])
ax.set_yticks(range(len(mode_pct_pivot_sorted)))
ax.set_yticklabels(mode_pct_pivot_sorted.index, fontsize=7)
plt.colorbar(im, ax=ax, label='Mode prevalence (fraction of ticks)')
ax.set_title('Mode prevalence heatmap — 50 products × 3 days\n(green=high pinning, red=low)')
plt.tight_layout()
plt.savefig(plots_dir / '01_mode_prevalence_heatmap.png', dpi=100)
plt.show()
plt.close()

print('Top 10 products by mean mode prevalence:')
print(mode_pct_avg.head(10).round(4).to_string())
print()
print('Bottom 10 products by mean mode prevalence:')
print(mode_pct_avg.tail(10).round(4).to_string())

Top 10 products by mean mode prevalence:
product
ROBOT_DISHES                   0.0563
OXYGEN_SHAKE_EVENING_BREATH    0.0419
ROBOT_IRONING                  0.0339
OXYGEN_SHAKE_CHOCOLATE         0.0112
SNACKPACK_PISTACHIO            0.0046
SNACKPACK_VANILLA              0.0039
SNACKPACK_CHOCOLATE            0.0038
SNACKPACK_STRAWBERRY           0.0036
SNACKPACK_RASPBERRY            0.0036
TRANSLATOR_GRAPHITE_MIST       0.0034

Bottom 10 products by mean mode prevalence:
product
MICROCHIP_TRIANGLE               0.0021
SLEEP_POD_SUEDE                  0.0020
PANEL_2X4                        0.0020
GALAXY_SOUNDS_PLANETARY_RINGS    0.0020
MICROCHIP_OVAL                   0.0019
OXYGEN_SHAKE_GARLIC              0.0018
MICROCHIP_SQUARE                 0.0018
SLEEP_POD_COTTON                 0.0018
PEBBLES_XS                       0.0015
PEBBLES_XL                       0.0012


## Cell 12 — Price level comparison across products within each category

Do products within a category share the same price level, or are they at different levels?

In [12]:
mean_all = all_df.groupby('product')['mid_price'].mean().reset_index()
mean_all.columns = ['product', 'mean_mid']
mean_all['category'] = mean_all['product'].map(product_to_cat)

print('=== Mean mid-price per product, grouped by category ===')
for cat in sorted(CATEGORIES.keys()):
    sub = mean_all[mean_all['category']==cat].sort_values('mean_mid')
    vals = sub['mean_mid'].values
    spread_within = vals.max() - vals.min() if len(vals) > 1 else 0
    print(f'\n{cat}  (within-cat spread: {spread_within:.2f})')
    for _, row in sub.iterrows():
        print(f"  {row['product']:<45} {row['mean_mid']:>10.4f}")

# Plot: category mean-price scatter
fig, ax = plt.subplots(figsize=(14, 6))
cat_list = sorted(CATEGORIES.keys())
colors_cat = cm.tab10(np.linspace(0, 1, len(cat_list)))
for i, (cat, color) in enumerate(zip(cat_list, colors_cat)):
    prods = mean_all[mean_all['category']==cat]
    ax.scatter([i]*len(prods), prods['mean_mid'], color=color, s=60, label=cat, zorder=3)
ax.set_xticks(range(len(cat_list)))
ax.set_xticklabels(cat_list, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Mean mid-price (all 3 days)')
ax.set_title('Mean mid-price per product, by category')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / '01_price_levels_by_category.png', dpi=100)
plt.show()
plt.close()

=== Mean mid-price per product, grouped by category ===

GALAXY_SOUNDS  (within-cat spread: 1240.21)
  GALAXY_SOUNDS_DARK_MATTER                     10226.6618
  GALAXY_SOUNDS_SOLAR_WINDS                     10437.5440
  GALAXY_SOUNDS_PLANETARY_RINGS                 10766.6732
  GALAXY_SOUNDS_SOLAR_FLAMES                    11092.5717
  GALAXY_SOUNDS_BLACK_HOLES                     11466.8721

MICROCHIP  (within-cat spread: 5415.15)
  MICROCHIP_OVAL                                 8179.5987
  MICROCHIP_RECTANGLE                            8732.4394
  MICROCHIP_CIRCLE                               9214.8853
  MICROCHIP_TRIANGLE                             9686.3911
  MICROCHIP_SQUARE                              13594.7483

OXYGEN_SHAKE  (within-cat spread: 2653.74)
  OXYGEN_SHAKE_EVENING_BREATH                    9271.8950
  OXYGEN_SHAKE_CHOCOLATE                         9556.8791
  OXYGEN_SHAKE_MINT                              9838.3941
  OXYGEN_SHAKE_MORNING_BREATH                  

## Cell 13 — Detailed FV check: integer vs half-integer prices, step grid analysis

Examine whether prices fall on a regular grid (e.g., integers only, or integer multiples of 0.5).

In [13]:
def grid_analysis(grp):
    vals = grp['mid_price'].dropna().values
    unique_vals = np.unique(vals)
    # Check if all values are integers
    all_int = np.all(unique_vals == np.round(unique_vals))
    # Check if all values are multiples of 0.5
    all_half = np.all((unique_vals * 2) == np.round(unique_vals * 2))
    # Minimum gap between consecutive distinct prices
    if len(unique_vals) > 1:
        diffs = np.diff(np.sort(unique_vals))
        min_gap = diffs.min()
        median_gap = np.median(diffs)
    else:
        min_gap = np.nan
        median_gap = np.nan
    return pd.Series({
        'all_int'    : all_int,
        'all_half'   : all_half,
        'min_gap'    : min_gap,
        'median_gap' : median_gap,
        'n_distinct' : len(unique_vals),
        'price_min'  : unique_vals.min(),
        'price_max'  : unique_vals.max(),
    })

grid = all_df.groupby('product').apply(grid_analysis, include_groups=False).reset_index()
print('=== Grid analysis — all 50 products (all days combined) ===')
print(grid.sort_values('n_distinct').to_string(index=False))

print(f"\nProducts where all mid_prices are integers: {grid[grid['all_int']]['product'].tolist()}")
print(f"Products where all mid_prices are half-integers: {grid[grid['all_half']]['product'].tolist()}")

=== Grid analysis — all 50 products (all days combined) ===
                      product  all_int  all_half  min_gap  median_gap  n_distinct  price_min  price_max
  OXYGEN_SHAKE_EVENING_BREATH    False      True      2.5         3.0         453     8277.5    10300.0
                ROBOT_IRONING    False      True      1.5         2.0         631     7458.5    10102.0
          SNACKPACK_RASPBERRY    False      True      0.5         0.5        1634     9682.5    10647.0
            SNACKPACK_VANILLA    False      True      0.5         0.5        1673     9675.5    10563.0
          SNACKPACK_CHOCOLATE    False      True      0.5         0.5        1900     9268.0    10315.5
          SNACKPACK_PISTACHIO    False      True      0.5         0.5        1928     9012.5    10106.5
    GALAXY_SOUNDS_DARK_MATTER    False      True      0.5         0.5        2907     9541.0    11186.0
                 ROBOT_DISHES    False      True      0.5         0.5        3048     8855.0    11400.0
    

## Cell 14 — Summary triage table

Combine all lenses into a single per-product summary table.

In [14]:
# Build summary dataframe
summary = fv[['product', 'fv_candidate', 'fv_pct', 'hardcoded_FV']].copy()
summary = summary.merge(avg_mode_pct.rename('avg_mode_pct').reset_index(), on='product')
summary = summary.merge(cv_avg.rename('avg_cv').reset_index(), on='product')
summary = summary.merge(rng_avg.rename('avg_range_pct').reset_index(), on='product')
summary = summary.merge(distinct_all.rename('n_distinct_all').reset_index(), on='product')
summary['category'] = summary['product'].map(product_to_cat)

# Triage label (distributional lens only)
def distrib_triage(row):
    if row['fv_pct'] >= 0.90:
        return 'likely exploitable'
    elif row['avg_mode_pct'] >= 0.50 or row['avg_cv'] < 0.001:
        return 'probably tradable'
    else:
        return 'needs corroboration'

summary['distrib_triage'] = summary.apply(distrib_triage, axis=1)

print('=== Per-product distributional summary (sorted by fv_pct desc) ===')
cols = ['product','category','fv_candidate','fv_pct','avg_mode_pct','avg_cv','avg_range_pct','n_distinct_all','distrib_triage']
print(summary.sort_values('fv_pct', ascending=False)[cols].to_string(index=False))

print()
print('Triage counts:')
print(summary['distrib_triage'].value_counts().to_string())

=== Per-product distributional summary (sorted by fv_pct desc) ===
                      product      category  fv_candidate   fv_pct  avg_mode_pct   avg_cv  avg_range_pct  n_distinct_all      distrib_triage
                 ROBOT_DISHES         ROBOT       10600.0 0.053800      0.056267 0.028054       0.121626            3048 needs corroboration
  OXYGEN_SHAKE_EVENING_BREATH  OXYGEN_SHAKE       10100.0 0.022733      0.041900 0.038204       0.160174             453 needs corroboration
                ROBOT_IRONING         ROBOT       10000.0 0.017700      0.033867 0.048613       0.180254             631 needs corroboration
       OXYGEN_SHAKE_CHOCOLATE  OXYGEN_SHAKE        9250.0 0.006467      0.011233 0.039102       0.155311            3512 needs corroboration
            SNACKPACK_VANILLA     SNACKPACK        9996.5 0.002833      0.003900 0.016011       0.077204            1673 needs corroboration
          SNACKPACK_CHOCOLATE     SNACKPACK       10005.5 0.002733      0.003833 0.0157

## Cell 15 — Day-over-day mean shift: is the hardcoded FV value stable across days?

In [15]:
# Compute mean mid per product per day; check if shift > some threshold
mean_pivot2 = all_df.groupby(['product','day'])['mid_price'].mean().unstack('day')
mean_pivot2['shift_d2_d3'] = (mean_pivot2[3] - mean_pivot2[2]).abs()
mean_pivot2['shift_d3_d4'] = (mean_pivot2[4] - mean_pivot2[3]).abs()
mean_pivot2['max_shift']   = mean_pivot2[['shift_d2_d3','shift_d3_d4']].max(axis=1)

print('=== Day-over-day mean shift (|day3-day2| and |day4-day3|) ===')
print(mean_pivot2.sort_values('max_shift', ascending=False).round(4).to_string())

# Flag products with stable means (shift < 10 absolute AND < 0.1% relative)
mean_all_combined = all_df.groupby('product')['mid_price'].mean()
mean_pivot2['max_shift_pct'] = mean_pivot2['max_shift'] / mean_all_combined
stable = mean_pivot2[mean_pivot2['max_shift_pct'] < 0.001]
print(f"\nProducts with max day-shift < 0.1% of mean (very stable FV): {len(stable)}")
print(stable[['max_shift','max_shift_pct']].round(6).to_string())

=== Day-over-day mean shift (|day3-day2| and |day4-day3|) ===
day                                     2           3           4  shift_d2_d3  shift_d3_d4  max_shift
product                                                                                               
MICROCHIP_SQUARE               11268.4607  14584.2410  14931.5432    3315.7803     347.3023  3315.7803
MICROCHIP_OVAL                  9766.3822   8543.7736   6228.6403    1222.6087    2315.1333  2315.1333
PEBBLES_XS                      9189.4057   6968.4102   6056.1034    2220.9955     912.3067  2220.9955
PEBBLES_XL                     11549.7666  13462.0964  14664.9048    1912.3298    1202.8084  1912.3298
SLEEP_POD_SUEDE                10254.9060  11899.1903  12038.1650    1644.2843     138.9747  1644.2843
MICROCHIP_TRIANGLE             10216.4920  10229.0848   8613.5964      12.5928    1615.4885  1615.4885
GALAXY_SOUNDS_PLANETARY_RINGS  10013.2524  11615.2732  10671.4940    1602.0208     943.7792  1602.0208
GALAXY_SOUN


Products with max day-shift < 0.1% of mean (very stable FV): 0
Empty DataFrame
Columns: [max_shift, max_shift_pct]
Index: []


## Cell 16 — Final summary printout and notebook conclusion

In [16]:
print('='*70)
print('NOTEBOOK 01 — DISTRIBUTIONAL ANALYSIS COMPLETE')
print('='*70)
print()
print('Key counts:')
print(f"  Total ticks analysed   : {len(all_df):,}")
print(f"  Products               : {all_df['product'].nunique()}")
print(f"  Days                   : {sorted(all_df['day'].unique())}")
print()
print('Hardcoded-FV summary (fv_pct >= 0.90 on combined days):')
hfv = fv[fv['hardcoded_FV']].sort_values('fv_pct', ascending=False)
if len(hfv):
    print(hfv[['product','fv_candidate','fv_pct','n_ticks']].to_string(index=False))
else:
    print('  None.')
print()
print('High mode prevalence (avg_mode_pct >= 0.50):')
hm = summary[summary['avg_mode_pct'] >= 0.50].sort_values('avg_mode_pct', ascending=False)
if len(hm):
    print(hm[['product','fv_candidate','avg_mode_pct','avg_cv']].to_string(index=False))
else:
    print('  None.')
print()
print('Lowest CV (most tightly pinned):')
print(summary.nsmallest(10, 'avg_cv')[['product','avg_cv','avg_range_pct']].to_string(index=False))
print()
print('Highest CV (most dispersed):')
print(summary.nlargest(10, 'avg_cv')[['product','avg_cv','avg_range_pct']].to_string(index=False))

NOTEBOOK 01 — DISTRIBUTIONAL ANALYSIS COMPLETE

Key counts:
  Total ticks analysed   : 1,500,000


  Products               : 50
  Days                   : [np.int64(2), np.int64(3), np.int64(4)]

Hardcoded-FV summary (fv_pct >= 0.90 on combined days):
  None.

High mode prevalence (avg_mode_pct >= 0.50):
  None.

Lowest CV (most tightly pinned):
                  product   avg_cv  avg_range_pct
      SNACKPACK_PISTACHIO 0.014482       0.065512
      SNACKPACK_CHOCOLATE 0.015784       0.078006
        SNACKPACK_VANILLA 0.016011       0.077204
      SNACKPACK_RASPBERRY 0.016572       0.081856
     SNACKPACK_STRAWBERRY 0.018081       0.087438
            ROBOT_LAUNDRY 0.024670       0.116782
             UV_VISOR_RED 0.025426       0.126635
          ROBOT_VACUUMING 0.027799       0.125868
             ROBOT_DISHES 0.028054       0.121626
GALAXY_SOUNDS_DARK_MATTER 0.029144       0.134200

Highest CV (most dispersed):
            product   avg_cv  avg_range_pct
         PEBBLES_XL 0.089695       0.342555
         PEBBLES_XS 0.082758       0.338128
     MICROCHIP_OVAL 0.063565       0.2